In [149]:
pip install reverse_geocoder --break-system-packages


Note: you may need to restart the kernel to use updated packages.


In [148]:
pip install osmnx geopandas pyproj shapely

Note: you may need to restart the kernel to use updated packages.


## **Pronze Layer**

In [73]:
import pandas as pd
import glob
import os
from sqlalchemy import create_engine
# الاتصال بالـ SQL Server
server = 'MALOKA\\SQLEXPRESS'
database = 'AirbnbDWH'

engine = create_engine(
    f'mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
)

# مسار الفولدر اللي فيه الـ 20 ملف
folder_path = r'C:\Users\DELL\Desktop\mozakra\Instant Task\archive (20)'
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

for file in all_files:
    filename = os.path.basename(file).replace('.csv', '')  # e.g. amsterdam_weekdays
    table_name = f'bronze_{filename}'  # e.g. bronze_amsterdam_weekdays
    
    df = pd.read_csv(file)
    
    df.to_sql(table_name, con=engine, if_exists='replace', index=False)
    print(f'Loaded: {table_name} ({df.shape[0]} rows)')

c:\Users\DELL\anaconda3\anacond\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())


Loaded: bronze_amsterdam_weekdays (1103 rows)
Loaded: bronze_amsterdam_weekends (977 rows)
Loaded: bronze_athens_weekdays (2653 rows)
Loaded: bronze_athens_weekends (2627 rows)
Loaded: bronze_barcelona_weekdays (1555 rows)
Loaded: bronze_barcelona_weekends (1278 rows)
Loaded: bronze_berlin_weekdays (1284 rows)
Loaded: bronze_berlin_weekends (1200 rows)
Loaded: bronze_budapest_weekdays (2074 rows)
Loaded: bronze_budapest_weekends (1948 rows)
Loaded: bronze_lisbon_weekdays (2857 rows)
Loaded: bronze_lisbon_weekends (2906 rows)
Loaded: bronze_london_weekdays (4614 rows)
Loaded: bronze_london_weekends (5379 rows)
Loaded: bronze_paris_weekdays (3130 rows)
Loaded: bronze_paris_weekends (3558 rows)
Loaded: bronze_rome_weekdays (4492 rows)
Loaded: bronze_rome_weekends (4535 rows)
Loaded: bronze_vienna_weekdays (1738 rows)
Loaded: bronze_vienna_weekends (1799 rows)


#### to be sure that no null values in the tables

In [74]:
server = 'MALOKA\\SQLEXPRESS'
database = 'AirbnbDWH'

engine = create_engine(
    f'mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes'
)
# أسماء الـ 20 جدول (المدن + نوع اليوم)
cities = ['amsterdam', 'athens', 'barcelona', 'berlin', 'budapest',
            'lisbon', 'london', 'paris', 'rome', 'vienna']
day_types = ['weekdays', 'weekends']

for city in cities:
    for day_type in day_types:
        table_name = f'bronze_{city}_{day_type}'
        df = pd.read_sql(f'SELECT * FROM {table_name}', con=engine)
        
        null_counts = df.isnull().sum()
        nulls_only = null_counts[null_counts > 0]
        
        print(f'\n--- {table_name} ---')
        if nulls_only.empty:
            print(' no nulls')
        else:
            print(nulls_only)


--- bronze_amsterdam_weekdays ---
 no nulls

--- bronze_amsterdam_weekends ---
 no nulls

--- bronze_athens_weekdays ---
 no nulls

--- bronze_athens_weekends ---
 no nulls

--- bronze_barcelona_weekdays ---
 no nulls

--- bronze_barcelona_weekends ---
 no nulls

--- bronze_berlin_weekdays ---
 no nulls

--- bronze_berlin_weekends ---
 no nulls

--- bronze_budapest_weekdays ---
 no nulls

--- bronze_budapest_weekends ---
 no nulls

--- bronze_lisbon_weekdays ---
 no nulls

--- bronze_lisbon_weekends ---
 no nulls


c:\Users\DELL\anaconda3\anacond\Lib\site-packages\pandas\io\sql.py:1636: SAWarning: Unrecognized server version info '17.0.1000.7'.  Some SQL Server features may not function properly.
  con = self.exit_stack.enter_context(con.connect())



--- bronze_london_weekdays ---
 no nulls

--- bronze_london_weekends ---
 no nulls

--- bronze_paris_weekdays ---
 no nulls

--- bronze_paris_weekends ---
 no nulls

--- bronze_rome_weekdays ---
 no nulls

--- bronze_rome_weekends ---
 no nulls

--- bronze_vienna_weekdays ---
 no nulls

--- bronze_vienna_weekends ---
 no nulls


## **Silver Layer**

### **1. Merge the essiential data**
#### (Airbnb Prices in European Cities)
https://www.kaggle.com/datasets/thedevastator/airbnb-prices-in-european-cities 

In [75]:
folder_path = r'C:\Users\DELL\Desktop\mozakra\Instant Task\archive (20)'
all_files = glob.glob(os.path.join(folder_path, "*.csv"))

df_list = []

for file in all_files:
    filename = os.path.basename(file).replace('.csv', '')
    city, day_type = filename.split('_')
    
    df = pd.read_csv(file)
    
    df['city'] = city
    df['day_type'] = day_type
    
    df_list.append(df)

# Merge 
Silver_df = pd.concat(df_list, ignore_index=True)


In [76]:
Silver_df.isna().sum()

Unnamed: 0                    0
realSum                       0
room_type                     0
room_shared                   0
room_private                  0
person_capacity               0
host_is_superhost             0
multi                         0
biz                           0
cleanliness_rating            0
guest_satisfaction_overall    0
bedrooms                      0
dist                          0
metro_dist                    0
attr_index                    0
attr_index_norm               0
rest_index                    0
rest_index_norm               0
lng                           0
lat                           0
city                          0
day_type                      0
dtype: int64

In [77]:
Silver_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 22 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Unnamed: 0                  51707 non-null  int64  
 1   realSum                     51707 non-null  float64
 2   room_type                   51707 non-null  object 
 3   room_shared                 51707 non-null  bool   
 4   room_private                51707 non-null  bool   
 5   person_capacity             51707 non-null  float64
 6   host_is_superhost           51707 non-null  bool   
 7   multi                       51707 non-null  int64  
 8   biz                         51707 non-null  int64  
 9   cleanliness_rating          51707 non-null  float64
 10  guest_satisfaction_overall  51707 non-null  float64
 11  bedrooms                    51707 non-null  int64  
 12  dist                        51707 non-null  float64
 13  metro_dist                  517

In [139]:
df.drop(columns=['Unnamed: 0'],inplace=True)

#### **From (Lng , Lat)  by using reverse_geocoder** :
 Extract this Features (city , region , sub region )


In [135]:
import pandas as pd
import reverse_geocoder as rg

df = Silver_df.copy()

# تجهيز الإحداثيات كـ list of tuples (lat, lng)
coordinates = list(zip(df['lat'], df['lng']))

# استعلام واحد بس لكل الصفوف (بيرجع النتيجة فورًا، محلي بالكامل)
results = rg.search(coordinates)  # list of dicts

df['region'] = [r['admin1'] for r in results]      # المحافظة/الولاية
df['sub_region'] = [r['admin2'] for r in results]      # المنطقة الفرعية


In [ ]:
df['distance_score'] = 1 / (1 + df['dist'] + df['metro_dist'])
df['price_per_person'] = df['realSum'] / df['person_capacity']
df['host_is_superhost_num'] = df['host_is_superhost'].astype(int)

df['cleanliness_norm'] = (df['cleanliness_rating'] - df['cleanliness_rating'].min()) / \
                            (df['cleanliness_rating'].max() - df['cleanliness_rating'].min())
df['satisfaction_norm'] = (df['guest_satisfaction_overall'] - df['guest_satisfaction_overall'].min()) / \
                            (df['guest_satisfaction_overall'].max() - df['guest_satisfaction_overall'].min())

df['luxury_score'] = (
    df['cleanliness_norm'].fillna(0) * 0.4 +
    df['satisfaction_norm'].fillna(0) * 0.4 +
    df['host_is_superhost_num'] * 0.2
)

In [140]:
df.isna().sum()

realSum                       0
room_type                     0
room_shared                   0
room_private                  0
person_capacity               0
host_is_superhost             0
multi                         0
biz                           0
cleanliness_rating            0
guest_satisfaction_overall    0
bedrooms                      0
dist                          0
metro_dist                    0
attr_index                    0
attr_index_norm               0
rest_index                    0
rest_index_norm               0
lng                           0
lat                           0
city                          0
day_type                      0
region                        0
sub_region                    0
dtype: int64

### 2.Merge Data
####  European City - Population and Area 
https://www.kaggle.com/datasets/rathiankit/european-city-population-and-area?utm_source=chatgpt.com 

In [85]:
population_df = pd.read_excel(r"C:\Users\DELL\Desktop\mozakra\Instant Task\European City - Population and Area\PopulatinandArea.xlsx")

In [ ]:
df["city"] = df["city"].str.strip().str.lower()
population_df["City"] = population_df["City"].str.strip().str.lower()

In [87]:
population_df.rename(columns={"City": "city"}, inplace=True)

In [143]:
df = df.merge(
    population_df,
    left_on="city",
    right_on="city",
    how="left"
)

#### to Extract : Cafes , Restaurants , Hotels , Hospitals 

In [150]:
import osmnx as ox
import pandas as pd

cities = {
    "amsterdam": "Amsterdam, Netherlands",
    "athens": "Athens, Greece",
    "barcelona": "Barcelona, Spain",
    "berlin": "Berlin, Germany",
    "budapest": "Budapest, Hungary",
    "lisbon": "Lisbon, Portugal",
    "london": "London, United Kingdom",
    "paris": "Paris, France",
    "rome": "Rome, Italy",
    "vienna": "Vienna, Austria"
}

def get_pois(tags):

    all_pois = []

    for city_key, city_name in cities.items():

        try:

            pois = ox.features_from_place(
                city_name,
                tags
            )

            pois = pois.reset_index()

            pois["city"] = city_key

            pois["lat"] = pois.geometry.centroid.y
            pois["lng"] = pois.geometry.centroid.x

            cols = ["city", "lat", "lng"]

            if "name" in pois.columns:
                cols.insert(1, "name")

            all_pois.append(
                pois[cols]
            )

            print(f"Done {city_key}")

        except Exception as e:

            print(city_key, e)

    return pd.concat(
        all_pois,
        ignore_index=True
    )

In [151]:
restaurants = get_pois(
    {"amenity": "restaurant"}
)

restaurants.head()

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x


Done amsterdam
athens Nominatim did not geocode query 'Athens, Greece' to a geometry of type (Multi)Polygon.


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x


Done barcelona


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from

...

Done paris


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x


Done rome
Done vienna


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x


,city,name,lat,lng
0,amsterdam,De Roode Leeuw,52.373759,4.893610
1,amsterdam,Turbo,52.360645,4.925313
2,amsterdam,Meram,52.356011,4.925595
3,amsterdam,Café Maxwell,52.356802,4.918653
4,amsterdam,ThaiCoon,52.357137,4.917658


In [152]:
cafes = get_pois({"amenity": "cafe"})

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from

Done amsterdam
athens Nominatim did not geocode query 'Athens, Greece' to a geometry of type (Multi)Polygon.
Done barcelona


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from

Done berlin
Done budapest


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x


...

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y


Done london


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x


Done paris
Done rome
Done vienna


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from

In [153]:
hotels = get_pois({"tourism": "hotel"})

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from

Done amsterdam
athens Nominatim did not geocode query 'Athens, Greece' to a geometry of type (Multi)Polygon.
Done barcelona


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from

Done berlin
Done budapest


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x


Done lisbon


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from

Done london
Done paris
Done rome
Done vienna


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lng"] = pois.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:34: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["lat"] = pois.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\528390078.py:35: UserWarning: Geometry is in a geographic CRS. Results from

In [97]:
attractions = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\attractions.csv")
restaurants = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\restaurants.csv")
cafes = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\cafes.csv")
hotels = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\hotels.csv")

In [98]:
import numpy as np
from sklearn.neighbors import BallTree

EARTH_RADIUS = 6371

def add_location_features(
    airbnb_df,
    poi_df,
    prefix,
    radius_km=1
):

    airbnb_df[f'{prefix}_count'] = 0
    airbnb_df[f'nearest_{prefix}_km'] = np.nan
    airbnb_df[f'nearest_{prefix}_name'] = None

    for city in airbnb_df['city'].unique():

        airbnb_city = airbnb_df[
            airbnb_df['city'] == city
        ]

        poi_city = poi_df[
            poi_df['city'].str.lower() == city
        ]

        if len(poi_city) == 0:
            continue

        tree = BallTree(
            np.radians(
                poi_city[['lat','lng']].values
            ),
            metric='haversine'
        )

        coords = np.radians(
            airbnb_city[['lat','lng']].values
        )

        radius = radius_km / EARTH_RADIUS

        counts = tree.query_radius(
            coords,
            r=radius,
            count_only=True
        )

        distances, indices = tree.query(
            coords,
            k=1
        )

        airbnb_df.loc[
            airbnb_city.index,
            f'{prefix}_count'
        ] = counts

        airbnb_df.loc[
            airbnb_city.index,
            f'nearest_{prefix}_km'
        ] = distances.flatten() * EARTH_RADIUS

        if 'name' in poi_city.columns:

            nearest_names = (
                poi_city
                .iloc[indices.flatten()]
                ['name']
                .values
            )

            airbnb_df.loc[
                airbnb_city.index,
                f'nearest_{prefix}_name'
            ] = nearest_names

    return airbnb_df

In [155]:
df = add_location_features(
    df,
    attractions,
    "attraction",
    radius_km=3
)

In [156]:
df = add_location_features(
    df,
    restaurants,
    "restaurant",
    radius_km=3
)

In [157]:
df = add_location_features(
    df,
    cafes,
    "cafe",
    radius_km=1
)

In [158]:
df = add_location_features(
    df,
    hotels,
    "hotel",
    radius_km=1
)

In [159]:
df.isna().sum()

realSum                           0
room_type                         0
room_shared                       0
room_private                      0
person_capacity                   0
host_is_superhost                 0
multi                             0
biz                               0
cleanliness_rating                0
guest_satisfaction_overall        0
bedrooms                          0
dist                              0
metro_dist                        0
attr_index                        0
attr_index_norm                   0
rest_index                        0
rest_index_norm                   0
lng                               0
lat                               0
city                              0
day_type                          0
region                            0
sub_region                        0
CityNo                            0
Country                           0
Population                        0
Area in km2                       0
attraction_count            

In [104]:
print(df['city'].unique())
print("________________________________________________________________________________")
print(restaurants['city'].unique())
print("________________________________________________________________________________")
print(cafes['city'].unique())
print("________________________________________________________________________________")
print(hotels['city'].unique())

['amsterdam' 'athens' 'barcelona' 'berlin' 'budapest' 'lisbon' 'london'
 'paris' 'rome' 'vienna']
________________________________________________________________________________
['amsterdam' 'barcelona' 'berlin' 'budapest' 'lisbon' 'london' 'paris'
 'rome' 'vienna']
________________________________________________________________________________
['amsterdam' 'barcelona' 'berlin' 'budapest' 'lisbon' 'london' 'paris'
 'rome' 'vienna']
________________________________________________________________________________
['amsterdam' 'barcelona' 'berlin' 'budapest' 'lisbon' 'london' 'paris'
 'rome']


In [106]:
from sklearn.neighbors import BallTree
import numpy as np

def find_nearest_landmark(listings_df, landmarks_df, listing_lat_col, listing_lng_col,
                            landmark_lat_col, landmark_lng_col, landmark_name_col):
    
    # تحويل الإحداثيات لـ radians (شرط أساسي لـ haversine)
    listings_rad = np.radians(listings_df[[listing_lat_col, listing_lng_col]].values)
    landmarks_rad = np.radians(landmarks_df[[landmark_lat_col, landmark_lng_col]].values)
    
    tree = BallTree(landmarks_rad, metric='haversine')
    
    # query() بترجع دايماً أقرب نقطة، من غير حد أقصى للمسافة
    distances, indices = tree.query(listings_rad, k=1)
    
    # تحويل من radians لـ km (نصف قطر الأرض ≈ 6371 كم)
    distances_km = distances.flatten() * 6371
    
    nearest_names = landmarks_df.iloc[indices.flatten()][landmark_name_col].values
    
    return distances_km, nearest_names

In [ ]:
df['nearest_attraction_km'], df['nearest_attraction_name'] = find_nearest_landmark(
    df, attractions, 'lat', 'lng', 'lat', 'lng', 'name'
)

df['nearest_cafe_km'], df['nearest_cafe_name'] = find_nearest_landmark(
    df, cafes, 'lat', 'lng', 'lat', 'lng', 'name'
)

df['nearest_hotel_km'], df['nearest_hotel_name'] = find_nearest_landmark(
    df, hotels, 'lat', 'lng', 'lat', 'lng', 'name'
)

df['nearest_restaurant_km'], df['nearest_restaurant_name'] = find_nearest_landmark(
    df, restaurants, 'lat', 'lng', 'lat', 'lng', 'name'
)

In [ ]:
df.isna().sum()

realSum                           0
room_type                         0
room_shared                       0
room_private                      0
person_capacity                   0
host_is_superhost                 0
multi                             0
biz                               0
cleanliness_rating                0
guest_satisfaction_overall        0
bedrooms                          0
dist                              0
metro_dist                        0
attr_index                        0
attr_index_norm                   0
rest_index                        0
rest_index_norm                   0
lng                               0
lat                               0
city                              0
day_type                          0
region                            0
sub_region                        0
CityNo                            0
Country                           0
Population                        0
Area in km2                       0
attraction_count            

In [160]:
df['nearest_attraction_name'] = df['nearest_attraction_name'].fillna('Unknown')
df['nearest_restaurant_name'] = df['nearest_restaurant_name'].fillna('Unknown')
df['nearest_cafe_name'] = df['nearest_cafe_name'].fillna('Unknown')
df['nearest_hotel_name'] = df['nearest_hotel_name'].fillna('Unknown')

In [161]:
df["year"] = 2020

### Merge Data -----> European Tour Destinations Dataset
https://www.kaggle.com/datasets/faizadani/european-tour-destinations-dataset

In [163]:
tour_df = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\European Tour Destinations Dataset\destinations.csv",
    encoding="latin1")
print(tour_df.columns.tolist())

['Destination', 'Region', 'Country', 'Category', 'Latitude', 'Longitude', 'Approximate Annual Tourists', 'Currency', 'Majority Religion', 'Famous Foods', 'Language', 'Best Time to Visit', 'Cost of Living', 'Safety', 'Cultural Significance', 'Description']


In [164]:
cities = [
    'Amsterdam',
    'Athens',
    'Barcelona',
    'Berlin',
    'Budapest',
    'Lisbon',
    'London',
    'Paris',
    'Rome',
    'Vienna'
]

tour_df[
    tour_df['Destination'].isin(cities)
]

,Destination,Region,Country,Category,Latitude,Longitude,Approximate Annual Tourists,Currency,Majority Religion,Famous Foods,Language,Best Time to Visit,Cost of Living,Safety,Cultural Significance,Description
0,Rome,Lazio,Italy,City,41.902782,12.496366,14 million,Euro,Roman Catholic,"Pizza, Pasta, Gelato",Italian,Spring (April-May) or Fall (Sept-Oct),Medium-high,"Generally safe, but watch out for pickpockets","The capital city, known for its historical lan...","A hub of ancient history and modern culture, w..."
10,Barcelona,Catalonia,Spain,City,41.385064,2.173135,12.7 million,Euro,Roman Catholic,"Paella, Tapas, Gazpacho",Spanish,Spring (April-May) or Fall (Sept-Oct),Medium-high,"Generally safe, but watch out for pickpockets","Known for its Gaudi architecture, beautiful be...","A city of stunning architecture, art, and vibr..."
20,Paris,?le-de-France,France,City,48.856614,2.352222,35-40 million,Euro,Roman Catholic,"Croissants, Baguettes, Cheese",French,Spring (April-May) or Fall (Sept-Oct),High,"Generally safe, but watch out for pickpockets",Home to iconic landmarks like the Eiffel Tower...,"A city of art, fashion, and culture, featuring..."
30,Vienna,Vienna,Austria,City,48.208171,16.373869,7.5 million,Euro,Roman Catholic,"Wiener Schnitzel, Sachertorte, Apfelstrudel",German,Spring (April-May) or Fall (Sept-Oct),Medium-high,Generally safe,"Known for its imperial palaces, classical musi...","A cultural capital with imperial architecture,..."
60,Berlin,Berlin,Germany,City,52.520008,13.404954,13.5 million,Euro,Protestant,"Currywurst, Bratwurst, Sauerkraut",German,Spring (April-May) or Fall (Sept-Oct),Medium-high,"Generally safe, but watch for pickpockets",The capital city known for its historical land...,"Famous for the Brandenburg Gate, Berlin Wall, ..."
90,Athens,Attica,Greece,City,37.979452,23.716217,10 million,Euro,Greek Orthodox,"Gyros, Souvlaki, Moussaka",Greek,Spring (April-May) or Fall (Sept-Oct),Medium-high,"Generally safe, but watch out for pickpockets","Capital city known for ancient ruins, museums,...",NaN
100,Lisbon,Lisbon,Portugal,City,38.710825,-9.136136,3.5 million,Euro,Roman Catholic,"Bacalhau, Pastel de nata, Sardinha",Portuguese,Spring (April-May) or Fall (Sept-Oct),Medium-high,"Generally safe, but watch out for pickpockets","Capital city known for its historic center, Se...",NaN
160,London,Greater London,United Kingdom,City,51.507351,-0.127758,25 million,British Pound Sterling (GBP),Christian (Anglican),"Fish and chips, Sunday roast, Afternoon tea",English,Spring (April-May) or Fall (Sep-Oct),High,"Generally safe, but be aware of pickpockets","The capital city, known for its Buckingham Pal...","The capital city, known for its Buckingham Pal..."


In [115]:
tour_df.columns

Index(['Destination', 'Region', 'Country', 'Category', 'Latitude', 'Longitude',
       'Approximate Annual Tourists', 'Currency', 'Majority Religion',
       'Famous Foods', 'Language', 'Best Time to Visit', 'Cost of Living',
       'Safety', 'Cultural Significance', 'Description'],
      dtype='object')

In [165]:
tour_df = tour_df.rename(
    columns={
        'Destination':'city'
    }
)

tour_df['city'] = (
    tour_df['city']
    .str.lower()
    .str.strip()
)

df['city'] = (
    df['city']
    .str.lower()
    .str.strip()
)

In [166]:
df = df.merge(
    tour_df[
        [
            'city',
            'Approximate Annual Tourists',
            'Currency',
            'Majority Religion',
            'Famous Foods',
            'Language',
            'Best Time to Visit',
            'Cost of Living',
            'Safety',
            'Cultural Significance',
            'Description'
        ]
    ],
    on='city',
    how='left'
)

In [167]:
df.isna().sum()

realSum                            0
room_type                          0
room_shared                        0
room_private                       0
person_capacity                    0
host_is_superhost                  0
multi                              0
biz                                0
cleanliness_rating                 0
guest_satisfaction_overall         0
bedrooms                           0
dist                               0
metro_dist                         0
attr_index                         0
attr_index_norm                    0
rest_index                         0
rest_index_norm                    0
lng                                0
lat                                0
city                               0
day_type                           0
region                             0
sub_region                         0
CityNo                             0
Country                            0
Population                         0
Area in km2                        0
a

In [168]:
fill_values = {
    'amsterdam': {
        'Approximate Annual Tourists': '20 million',
        'Currency': 'Euro',
        'Majority Religion': 'Christianity (Catholic & Protestant)',
        'Famous Foods': 'Stroopwafel, Herring, Bitterballen',
        'Language': 'Dutch',
        'Best Time to Visit': 'Spring (Apr-May) or Summer (Jun-Aug)',
        'Cost of Living': 'High',
        'Safety': 'Generally safe, watch for pickpockets in tourist areas',
        'Cultural Significance': 'Capital of the Netherlands, famous for canals, museums, and cultural heritage',
        'Description': 'A vibrant European capital known for its historic canals, cycling culture, world-class museums, and lively atmosphere.'
    },

    'budapest': {
        'Approximate Annual Tourists': '12 million',
        'Currency': 'Hungarian Forint',
        'Majority Religion': 'Christianity (Roman Catholic)',
        'Famous Foods': 'Goulash, Chimney Cake, Langos',
        'Language': 'Hungarian',
        'Best Time to Visit': 'Spring (Apr-May) or Fall (Sep-Oct)',
        'Cost of Living': 'Medium',
        'Safety': 'Generally safe, exercise normal precautions',
        'Cultural Significance': 'Capital of Hungary, renowned for thermal baths, architecture, and Danube River landmarks',
        'Description': 'A historic city divided by the Danube River, famous for thermal baths, stunning architecture, and rich cultural traditions.'
    }
}

for city, vals in fill_values.items():
    mask = (df['city'] == city)

    for col, value in vals.items():
        df.loc[mask & df[col].isna(), col] = value

In [169]:
df[
    [
        'Approximate Annual Tourists',
        'Currency',
        'Majority Religion',
        'Famous Foods',
        'Language',
        'Best Time to Visit',
        'Cost of Living',
        'Safety',
        'Cultural Significance'
    ]
].isna().sum()

Approximate Annual Tourists    0
Currency                       0
Majority Religion              0
Famous Foods                   0
Language                       0
Best Time to Visit             0
Cost of Living                 0
Safety                         0
Cultural Significance          0
dtype: int64

In [170]:
df["Safety"].value_counts()

Safety
Generally safe, but watch out for pickpockets             29591
Generally safe, but be aware of pickpockets                9993
Generally safe, exercise normal precautions                4022
Generally safe                                             3537
Generally safe, but watch for pickpockets                  2484
Generally safe, watch for pickpockets in tourist areas     2080
Name: count, dtype: int64

In [171]:
safety_map = {
    'Generally safe': 'Very Safe',
    'Generally safe, exercise normal precautions': 'Safe',
    'Generally safe, but watch out for pickpockets': 'Safe',
    'Generally safe, but be aware of pickpockets': 'Safe',
    'Generally safe, but watch for pickpockets': 'Safe',
    'Generally safe, watch for pickpockets in tourist areas': 'Moderately Safe'
}

df['Safety'] = df['Safety'].replace(safety_map)

In [172]:
df.isna().sum()

realSum                            0
room_type                          0
room_shared                        0
room_private                       0
person_capacity                    0
host_is_superhost                  0
multi                              0
biz                                0
cleanliness_rating                 0
guest_satisfaction_overall         0
bedrooms                           0
dist                               0
metro_dist                         0
attr_index                         0
attr_index_norm                    0
rest_index                         0
rest_index_norm                    0
lng                                0
lat                                0
city                               0
day_type                           0
region                             0
sub_region                         0
CityNo                             0
Country                            0
Population                         0
Area in km2                        0
a

In [179]:
df['Description'] = df['Description'].fillna('No Description')

In [180]:
df['city_avg_rating'] = ( df.groupby('city')['guest_satisfaction_overall'].transform('mean'))
df['city_avg_price'] = (df.groupby('city')['realSum'].transform('mean'))

In [181]:
cost_map = {
    'Low':1,
    'Medium':2,
    'Medium-high':3,
    'High':4
}

df['cost_score'] = df['Cost of Living'].map(cost_map)
df['price_vs_cost_of_living'] = (df['realSum'] /df['cost_score'])

In [182]:
import osmnx as ox
import pandas as pd

cities = [
    "Amsterdam",
    "Athens",
    "Barcelona",
    "Berlin",
    "Budapest",
    "Lisbon",
    "London",
    "Paris",
    "Rome",
    "Vienna"
]

all_parks = []

for city in cities:

    try:

        parks = ox.features_from_place(
            city,
            {"leisure":"park"}
        )

        parks = parks.reset_index()

        parks["city"] = city.lower()

        parks["lat"] = parks.geometry.centroid.y
        parks["lng"] = parks.geometry.centroid.x

        all_parks.append(
            parks[["city","name","lat","lng"]]
        )

        print(city)

    except Exception as e:
        print(city, e)

parks = pd.concat(
    all_parks,
    ignore_index=True
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lat"] = parks.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lng"] = parks.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lat"] = parks.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:33: UserWarning: Geometry is in a geographic CRS. Re

Amsterdam
Athens


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lat"] = parks.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lng"] = parks.geometry.centroid.x


Barcelona


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lat"] = parks.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lng"] = parks.geometry.centroid.x


...

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lat"] = parks.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lng"] = parks.geometry.centroid.x


Paris


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lat"] = parks.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lng"] = parks.geometry.centroid.x


Rome
Vienna


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:32: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lat"] = parks.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\1689244989.py:33: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  parks["lng"] = parks.geometry.centroid.x


In [183]:
parks.to_csv("parks.csv", index=False)

In [184]:
all_hospitals = []

for city in cities:

    try:

        hospitals = ox.features_from_place(
            city,
            {"amenity":"hospital"}
        )

        hospitals = hospitals.reset_index()

        hospitals["city"] = city.lower()

        hospitals["lat"] = hospitals.geometry.centroid.y
        hospitals["lng"] = hospitals.geometry.centroid.x

        all_hospitals.append(
            hospitals[["city","name","lat","lng"]]
        )

        print(city)

    except Exception as e:
        print(city, e)

hospitals = pd.concat(
    all_hospitals,
    ignore_index=True
)

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lng"] = hospitals.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is

Amsterdam
Athens


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lng"] = hospitals.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is

Barcelona
Berlin


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lng"] = hospitals.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is

Budapest
Lisbon


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lng"] = hospitals.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is

London
Paris
Rome
Vienna


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lng"] = hospitals.geometry.centroid.x
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:16: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  hospitals["lat"] = hospitals.geometry.centroid.y
C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3743149858.py:17: UserWarning: Geometry is

In [185]:
hospitals.to_csv("hospitals.csv", index=False)

In [186]:
import numpy as np
from sklearn.neighbors import BallTree

def add_poi_features(df, poi_df, prefix, radius_km=3):

    poi_df = poi_df.dropna(
        subset=['lat', 'lng']
    ).copy()

    airbnb_coords = np.radians(
        df[['lat', 'lng']].values
    )

    poi_coords = np.radians(
        poi_df[['lat', 'lng']].values
    )

    tree = BallTree(
        poi_coords,
        metric='haversine'
    )

    radius = radius_km / 6371

    counts = tree.query_radius(
        airbnb_coords,
        r=radius,
        count_only=True
    )

    dist, ind = tree.query(
        airbnb_coords,
        k=1
    )

    df[f'{prefix}_count'] = counts

    df[f'nearest_{prefix}_km'] = (
        dist.flatten() * 6371
    )

    df[f'nearest_{prefix}_name'] = (
        poi_df.iloc[
            ind.flatten()
        ]['name'].values
    )

    return df

In [187]:
df = add_poi_features(
    df,
    parks,
    "park"
)

In [188]:
df = add_poi_features(
    df,
    hospitals,
    "hospital"
)

### 3. Merge Data : (Airport data)
from https://ourairports.com/data/?utm_source=chatgpt.com

In [206]:
air = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\airports.csv")
air

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85703,32753,ZYYY,medium_airport,Shenyang Dongta Airport,41.784354,123.496308,157.0,AS,CN,CN-21,"Dadong, Shenyang",no,ZYYY,NaN,ZYYY,NaN,NaN,NaN,"东塔机场, SHE"
85704,46378,ZZ-0001,heliport,Sealand Helipad,51.894444,1.482500,40.0,EU,GB,GB-ENG,Sealand,no,NaN,NaN,NaN,NaN,http://www.sealandgov.org/,https://en.wikipedia.org/wiki/Principality_of_...,Roughs Tower Helipad
85705,307326,ZZ-0002,small_airport,Glorioso Islands Airstrip,-11.584278,47.296389,11.0,AF,TF,TF-U-A,Grande Glorieuse,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85706,346788,ZZ-0003,small_airport,Fainting Goat Airport,32.110587,-97.356312,690.0,NaN,US,US-TX,Blum,no,NaN,NaN,87TX,87TX,NaN,NaN,NaN


In [207]:
air = air[
    [
        "name",
        "latitude_deg",
        "longitude_deg"
    ]
].copy()

air.rename(
    columns={
        "latitude_deg": "lat",
        "longitude_deg": "lng"
    },
    inplace=True
)

In [208]:
print(air.columns.tolist())

['name', 'lat', 'lng']


In [212]:
df = add_poi_features(
    df,
    air,
    "airport",
    radius_km=20
)

In [215]:
df.isna().sum()

realSum                    0
room_type                  0
room_shared                0
room_private               0
person_capacity            0
                        ... 
nearest_hospital_km        0
nearest_hospital_name    166
airport_count              0
nearest_airport_km         0
nearest_airport_name       0
Length: 63, dtype: int64

In [216]:
df[
    df['nearest_hospital_name'].isna()
][
    ['hospital_count', 'nearest_hospital_km']
].head()

,hospital_count,nearest_hospital_km
13136,18,1.334429
13308,18,1.205695
13452,13,1.022538
13487,0,3.032967
13880,0,3.134771


In [219]:
df["nearest_hospital_name"] = ( df["nearest_hospital_name"].fillna("No nearby hospital"))

In [261]:

safety_data = {
    'Amsterdam': {'Safety_Index': 70.2, 'Crime_Index': 29.8},
    'Athens':    {'Safety_Index': 45.0, 'Crime_Index': 55.0},
    'Barcelona': {'Safety_Index': 47.9, 'Crime_Index': 52.1	},
    'Berlin':    {'Safety_Index': 55.4, 'Crime_Index': 44.6},
    'Budapest':  {'Safety_Index': 66.5, 'Crime_Index': 33.5},
    'Lisbon':    {'Safety_Index': 32.9, 'Crime_Index': 67.1},
    'London':    {'Safety_Index': 44.8, 'Crime_Index': 55.2},
    'Paris':     {'Safety_Index': 42.1, 'Crime_Index': 57.9},
    'Rome':      {'Safety_Index': 53.3, 'Crime_Index': 46.7},
    'Vienna':    {'Safety_Index': 70.6, 'Crime_Index': 29.4	},
}

safety_df = pd.DataFrame.from_dict(safety_data, orient='index').reset_index()
safety_df.columns = ['city', 'Safety_Index', 'Crime_Index']

In [262]:
safety_df

,city,Safety_Index,Crime_Index
0,Amsterdam,70.2,29.8
1,Athens,45.0,55.0
2,Barcelona,47.9,52.1
3,Berlin,55.4,44.6
4,Budapest,66.5,33.5
5,Lisbon,32.9,67.1
6,London,44.8,55.2
7,Paris,42.1,57.9
8,Rome,53.3,46.7
9,Vienna,70.6,29.4


In [263]:
df['city'] = df['city'].str.lower()

safety_df['city'] = (
    safety_df['city']
    .str.lower()
)

In [264]:
df = df.merge(
    safety_df,
    on='city',
    how='left'
)

In [267]:
df.isna().sum()

realSum            0
room_type          0
room_shared        0
room_private       0
person_capacity    0
                  ..
rating_score       0
location_score     0
city_popularity    0
Safety_Index       0
Crime_Index        0
Length: 69, dtype: int64

In [246]:
df['amenities_score'] = (
    df['restaurant_count'] +
    df['cafe_count'] +
    df['park_count'] +
    df['hospital_count']
)

In [247]:
df['rating_score'] = (
    df['guest_satisfaction_overall'] +
    df['cleanliness_rating']
) / 2

In [248]:
df['location_score'] = (
    1/(df['dist']+1) +
    1/(df['metro_dist']+1) +
    1/(df['nearest_airport_km']+1)
)

In [249]:
df['city_popularity'] = (
    df['attraction_count'] +
    df['restaurant_count'] +
    df['cafe_count'] +
    df['hotel_count']
)

In [250]:
df.columns 

Index(['realSum', 'room_type', 'room_shared', 'room_private',
       'person_capacity', 'host_is_superhost', 'multi', 'biz',
       'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist',
       'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index',
       'rest_index_norm', 'lng', 'lat', 'city', 'day_type', 'region',
       'sub_region', 'CityNo', 'Country', 'Population', 'Area in km2 ',
       'attraction_count', 'nearest_attraction_km', 'nearest_attraction_name',
       'restaurant_count', 'nearest_restaurant_km', 'nearest_restaurant_name',
       'cafe_count', 'nearest_cafe_km', 'nearest_cafe_name', 'hotel_count',
       'nearest_hotel_km', 'nearest_hotel_name', 'year',
       'Approximate Annual Tourists', 'Currency', 'Majority Religion',
       'Famous Foods', 'Language', 'Best Time to Visit', 'Cost of Living',
       'Safety', 'Cultural Significance', 'Description', 'city_avg_rating',
       'city_avg_price', 'cost_score', 'price_vs_cost_of_living', 'park_c

In [ ]:
df["Best Time to Visit"]

0         Spring (Apr-May) or Summer (Jun-Aug)
1         Spring (Apr-May) or Summer (Jun-Aug)
2         Spring (Apr-May) or Summer (Jun-Aug)
3         Spring (Apr-May) or Summer (Jun-Aug)
4         Spring (Apr-May) or Summer (Jun-Aug)
                         ...                  
51702    Spring (April-May) or Fall (Sept-Oct)
51703    Spring (April-May) or Fall (Sept-Oct)
51704    Spring (April-May) or Fall (Sept-Oct)
51705    Spring (April-May) or Fall (Sept-Oct)
51706    Spring (April-May) or Fall (Sept-Oct)
Name: Best Time to Visit, Length: 51707, dtype: object

In [271]:
season_map = {
    'amsterdam': 'Spring',
    'athens': 'Spring',
    'barcelona': 'Spring/Fall',
    'berlin': 'Summer',
    'budapest': 'Spring/Fall',
    'lisbon': 'Spring/Fall',
    'london': 'Summer',
    'paris': 'Spring/Fall',
    'rome': 'Spring/Fall',
    'vienna': 'Spring/Fall'
}

df['Tourism_Season'] = df['city'].map(season_map)

In [272]:
months_map = {
    'amsterdam': 'Apr-Aug',
    'athens': 'Apr-Jun, Sep-Oct',
    'barcelona': 'Apr-Jun, Sep-Oct',
    'berlin': 'Jun-Aug',
    'budapest': 'Apr-Jun, Sep-Oct',
    'lisbon': 'Mar-May, Sep-Oct',
    'london': 'Jun-Aug',
    'paris': 'Apr-Jun, Sep-Oct',
    'rome': 'Apr-Jun, Sep-Oct',
    'vienna': 'Apr-Jun, Sep-Oct'
}

df['Best_Months'] = df['city'].map(months_map)

In [274]:
df.isna().sum()

realSum            0
room_type          0
room_shared        0
room_private       0
person_capacity    0
                  ..
city_popularity    0
Safety_Index       0
Crime_Index        0
Tourism_Season     0
Best_Months        0
Length: 71, dtype: int64

In [ ]:
df.drop(columns=['Best Time to Visit','CityNo'], inplace=True)

In [276]:
df.columns

Index(['realSum', 'room_type', 'room_shared', 'room_private',
       'person_capacity', 'host_is_superhost', 'multi', 'biz',
       'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist',
       'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index',
       'rest_index_norm', 'lng', 'lat', 'city', 'day_type', 'region',
       'sub_region', 'CityNo', 'Country', 'Population', 'Area in km2 ',
       'attraction_count', 'nearest_attraction_km', 'nearest_attraction_name',
       'restaurant_count', 'nearest_restaurant_km', 'nearest_restaurant_name',
       'cafe_count', 'nearest_cafe_km', 'nearest_cafe_name', 'hotel_count',
       'nearest_hotel_km', 'nearest_hotel_name', 'year',
       'Approximate Annual Tourists', 'Currency', 'Majority Religion',
       'Famous Foods', 'Language', 'Cost of Living', 'Safety',
       'Cultural Significance', 'Description', 'city_avg_rating',
       'city_avg_price', 'cost_score', 'price_vs_cost_of_living', 'park_count',
       'nearest

In [279]:
train = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\Airbnb Property Rental Price\train.csv")
test = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\Airbnb Property Rental Price\test.csv")

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3959213892.py:1: DtypeWarning: Columns (0,4,15,22,36,37,46) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(r"C:\Users\DELL\Desktop\mozakra\Instant Task\Airbnb Property Rental Price\train.csv")


In [280]:
print(train.shape)
print(test.shape)

print(train.columns.tolist())
print(test.columns.tolist())

(261895, 55)
(87303, 54)
['id', 'name', 'description', 'neighborhood_overview', 'host_id', 'host_name', 'host_since', 'host_location', 'host_about', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_neighbourhood', 'host_listings_count', 'host_total_listings_count', 'host_verifications', 'host_has_profile_pic', 'host_identity_verified', 'neighbourhood', 'neighbourhood_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price', 'has_availability', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'number_of_reviews', 'number_of_reviews_ltm', 'number_of_reviews_l30d', 'availability_eoy', 'number_of_reviews_ly', 'estimated_occupancy_l365d', 'estimated_revenue_l365d', 'first_review', 'last_review', 'review_scores_rating', 'review_scores_accuracy', 'review_scores_cleanliness', 'review_scores_checkin', 'review_scores_communicat

In [281]:
import pandas as pd

# إضافة price للـ test
test['price'] = pd.NA

# دمج الملفين
airbnb_new = pd.concat(
    [train, test],
    ignore_index=True
)

print(airbnb_new.shape)

(349198, 55)


C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3476709070.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  airbnb_new = pd.concat(


In [284]:
airbnb_new.isna().sum()

id                                  0
name                                1
description                      8088
neighborhood_overview          178875
host_id                             0
host_name                         159
host_since                        168
host_location                   89235
host_about                     168379
host_response_time              53759
host_response_rate              53759
host_acceptance_rate            30067
host_is_superhost                9911
host_neighbourhood             212127
host_listings_count               169
host_total_listings_count         169
host_verifications                169
host_has_profile_pic              169
host_identity_verified            169
neighbourhood                  178874
neighbourhood_cleansed          11599
latitude                            1
longitude                           1
property_type                       1
room_type                           1
accommodates                        1
bathrooms   

In [285]:
airbnb_new['city'] = (
    airbnb_new['city']
    .str.lower()
    .str.strip()
)

df['city'] = (
    df['city']
    .str.lower()
    .str.strip()
)

In [290]:
airbnb_new['city'].unique()

array(['albany', 'amsterdam', 'antwerp', 'asheville', 'athens', 'austin',
       'bangkok', 'barcelona', 'barossa valley', 'barwon south west, vic',
       'belize', 'bergamo', 'berlin', 'bologna', 'bordeaux', 'boston',
       'bozeman', 'brisbane', 'bristol', 'broward county', 'brussels',
       'cambridge', 'cape town', 'chicago', 'clark county, nv',
       'columbus', 'copenhagen', 'crete', 'dallas', 'denver', 'dublin',
       'edinburgh', 'euskadi', 'florence', 'fort worth', 'geneva',
       'ghent', 'girona', 'greater manchester', 'hawaii', 'hong kong',
       'ireland', 'istanbul', 'jersey city', 'lisbon', 'london',
       'los angeles', 'lyon', 'madrid', 'mallorca', 'malta', 'melbourne',
       'menorca', 'mexico city', 'mid north coast', 'milan', 'montreal',
       'mornington peninsula', 'munich', 'naples', 'nashville',
       'new brunswick', 'new orleans', 'new york city', 'newark',
       'northern rivers', 'oakland', 'oslo', 'ottawa', 'pacific grove',
       'paris', 'port

In [293]:
my_cities = [
    'amsterdam',
    'athens',
    'barcelona',
    'berlin',
    'budapest',
    'lisbon',
    'london',
    'paris',
    'rome',
    'vienna'
]

airbnb_new['city'] = (
    airbnb_new['city']
    .str.lower()
    .str.strip()
)

airbnb_filtered = airbnb_new[
    airbnb_new['city'].isin(my_cities)
].copy()

In [294]:
print(airbnb_filtered['city'].value_counts())

city
london       22520
paris        21344
lisbon        7575
barcelona     5491
athens        4959
vienna        3548
berlin        3246
amsterdam     2126
Name: count, dtype: int64


In [295]:
df['city'].value_counts()

city
london       9993
rome         9027
paris        6688
lisbon       5763
athens       5280
budapest     4022
vienna       3537
barcelona    2833
berlin       2484
amsterdam    2080
Name: count, dtype: int64

In [308]:
airbnb_filtered['latitude'] = pd.to_numeric(
    airbnb_filtered['latitude'],
    errors='coerce'
)

airbnb_filtered['longitude'] = pd.to_numeric(
    airbnb_filtered['longitude'],
    errors='coerce'
)

In [309]:
airbnb_filtered[['latitude', 'longitude']].dtypes

latitude     float64
longitude    float64
dtype: object

In [310]:
import pandas as pd
import numpy as np

df['lat'] = pd.to_numeric(df['lat'], errors='coerce')
df['lng'] = pd.to_numeric(df['lng'], errors='coerce')

airbnb_filtered['latitude'] = pd.to_numeric(
    airbnb_filtered['latitude'],
    errors='coerce'
)

airbnb_filtered['longitude'] = pd.to_numeric(
    airbnb_filtered['longitude'],
    errors='coerce'
)

airbnb_filtered = airbnb_filtered.dropna(
    subset=['latitude', 'longitude']
)

In [312]:
cols_to_merge = [
    'property_type',
    'accommodates',
    'bathrooms',
    'beds',
    'host_is_superhost',
    'host_response_rate',
    'host_acceptance_rate',
    'host_total_listings_count',
    'host_identity_verified',
    'availability_365',
    'number_of_reviews',
    'reviews_per_month',
    'review_scores_rating',
    'review_scores_location',
    'review_scores_cleanliness',
    'amenities'
]

In [313]:
from sklearn.neighbors import BallTree

all_matches = []

common_cities = sorted(
    set(df['city'].str.lower()) &
    set(airbnb_filtered['city'].str.lower())
)

for city in common_cities:

    old_city = df[
        df['city'].str.lower() == city
    ].copy()

    new_city = airbnb_filtered[
        airbnb_filtered['city'].str.lower() == city
    ].copy()

    if len(new_city) == 0:
        continue

    old_coords = np.radians(
        old_city[['lat', 'lng']].values.astype(float)
    )

    new_coords = np.radians(
        new_city[['latitude', 'longitude']].values.astype(float)
    )

    tree = BallTree(
        new_coords,
        metric='haversine'
    )

    dist, idx = tree.query(
        old_coords,
        k=1
    )

    matched = (
        new_city
        .iloc[idx.flatten()][cols_to_merge]
        .reset_index(drop=True)
    )

    matched.index = old_city.index

    all_matches.append(matched)

In [314]:
nearest_features = pd.concat(all_matches)

nearest_features = nearest_features.sort_index()

In [315]:
df = pd.concat(
    [
        df,
        nearest_features
    ],
    axis=1
)

In [322]:
df['review_scores_rating'] = pd.to_numeric(
    df['review_scores_rating'],
    errors='coerce'
)

                           Nulls  Percent    Dtype
review_scores_rating       19893    38.47  float64
review_scores_location     19893    38.47  float64
review_scores_cleanliness  19893    38.47  float64
host_response_rate         18124    35.05   object
host_acceptance_rate       16195    31.32   object
nearest_park_name          15001    29.01   object
host_is_superhost          14244    27.55   object
beds                       13148    25.43  float64
bathrooms                  13065    25.27  float64
host_total_listings_count  13051    25.24   object
host_identity_verified     13051    25.24   object
accommodates               13049    25.24  float64
reviews_per_month          13049    25.24  float64
number_of_reviews          13049    25.24   object
availability_365           13049    25.24   object
property_type              13049    25.24   object
amenities                  13049    25.24   object
nearest_hotel_km            5280    10.21  float64
nearest_restaurant_km       528

In [339]:
rating_cols = [
    'review_scores_rating',
    'review_scores_location',
    'review_scores_cleanliness',
    'reviews_per_month'
]

for col in rating_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

    df[col] = (
        df.groupby('city')[col]
          .transform(lambda x: x.fillna(x.mean()))
    )

    df[col] = df[col].fillna(df[col].mean())

In [340]:
rate_cols = [
    'host_response_rate',
    'host_acceptance_rate'
]

for col in rate_cols:

    df[col] = (
        df[col]
        .astype(str)
        .str.replace('%', '', regex=False)
    )

    df[col] = pd.to_numeric(
        df[col],
        errors='coerce'
    )

    df[col] = (
        df.groupby('city')[col]
          .transform(lambda x: x.fillna(x.median()))
    )

    df[col] = df[col].fillna(df[col].median())

In [341]:
property_cols = [
    'accommodates',
    'beds',
    'bathrooms',
    'availability_365',
    'number_of_reviews',
    'host_total_listings_count'
]

for col in property_cols:

    df[col] = pd.to_numeric(
        df[col],
        errors='coerce'
    )

    df[col] = (
        df.groupby('city')[col]
          .transform(lambda x: x.fillna(x.median()))
    )

    df[col] = df[col].fillna(df[col].median())

In [344]:
df['property_type'] = (
    df['property_type']
      .fillna(
          df.groupby('city')['property_type']
            .transform(lambda x: x.mode().iloc[0] if len(x.mode()) else 'Unknown')
      )
)

In [345]:
df['amenities_count'] = (
    df['amenities']
      .fillna('')
      .apply(lambda x: len(str(x).split(',')))
)

In [349]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 86 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   realSum                      51707 non-null  float64
 1   room_type                    51707 non-null  object 
 2   room_shared                  51707 non-null  bool   
 3   room_private                 51707 non-null  bool   
 4   person_capacity              51707 non-null  float64
 5   host_is_superhost            51707 non-null  bool   
 6   multi                        51707 non-null  int64  
 7   biz                          51707 non-null  int64  
 8   cleanliness_rating           51707 non-null  float64
 9   guest_satisfaction_overall   51707 non-null  float64
 10  bedrooms                     51707 non-null  int64  
 11  dist                         51707 non-null  float64
 12  metro_dist                   51707 non-null  float64
 13  attr_index      

In [351]:
df['wifi'] = df['amenities'].fillna('').str.contains(
    'wifi',
    case=False,
    regex=False
).astype(int)

df['parking'] = df['amenities'].fillna('').str.contains(
    'parking',
    case=False,
    regex=False
).astype(int)

df['kitchen'] = df['amenities'].fillna('').str.contains(
    'kitchen',
    case=False,
    regex=False
).astype(int)

df['air_conditioning'] = df['amenities'].fillna('').str.contains(
    'air conditioning',
    case=False,
    regex=False
).astype(int)

In [352]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51707 entries, 0 to 51706
Data columns (total 90 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   realSum                      51707 non-null  float64
 1   room_type                    51707 non-null  object 
 2   room_shared                  51707 non-null  bool   
 3   room_private                 51707 non-null  bool   
 4   person_capacity              51707 non-null  float64
 5   host_is_superhost            51707 non-null  bool   
 6   multi                        51707 non-null  int64  
 7   biz                          51707 non-null  int64  
 8   cleanliness_rating           51707 non-null  float64
 9   guest_satisfaction_overall   51707 non-null  float64
 10  bedrooms                     51707 non-null  int64  
 11  dist                         51707 non-null  float64
 12  metro_dist                   51707 non-null  float64
 13  attr_index      

In [354]:
df.columns

Index(['realSum', 'room_type', 'room_shared', 'room_private',
       'person_capacity', 'host_is_superhost', 'multi', 'biz',
       'cleanliness_rating', 'guest_satisfaction_overall', 'bedrooms', 'dist',
       'metro_dist', 'attr_index', 'attr_index_norm', 'rest_index',
       'rest_index_norm', 'lng', 'lat', 'city', 'day_type', 'region',
       'sub_region', 'Country', 'Population', 'Area in km2 ',
       'attraction_count', 'nearest_attraction_km', 'nearest_attraction_name',
       'restaurant_count', 'nearest_restaurant_km', 'nearest_restaurant_name',
       'cafe_count', 'nearest_cafe_km', 'nearest_cafe_name', 'hotel_count',
       'nearest_hotel_km', 'nearest_hotel_name', 'year',
       'Approximate Annual Tourists', 'Currency', 'Majority Religion',
       'Famous Foods', 'Language', 'Cost of Living', 'Safety',
       'Cultural Significance', 'Description', 'city_avg_rating',
       'city_avg_price', 'cost_score', 'price_vs_cost_of_living', 'park_count',
       'nearest_park_km',

In [364]:
df.duplicated().sum()

np.int64(0)

In [365]:
df = df.drop_duplicates()

In [366]:
df.to_csv('airbnb_merged data', index=False)

In [369]:
df = df.loc[:, ~df.columns.duplicated()]

(51707, 89)


3

In [373]:
df.drop(columns=['host_response_rate', 'host_acceptance_rate'], inplace=True)

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\3240127212.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(columns=['host_response_rate', 'host_acceptance_rate'], inplace=True)


In [374]:
for col in ['nearest_restaurant_km', 'nearest_cafe_km', 'nearest_hotel_km']:
    df[col] = df[col].fillna(df[col].median())

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\604574310.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[col] = df[col].fillna(df[col].median())


In [378]:
df['host_identity_verified'] = df['host_identity_verified'].fillna('Unknown')

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\570034684.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['host_identity_verified'] = df['host_identity_verified'].fillna('Unknown')


In [377]:
df['amenities'] = df['amenities'].fillna('No amenities listed')

C:\Users\DELL\AppData\Local\Temp\ipykernel_36464\2118866571.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['amenities'] = df['amenities'].fillna('No amenities listed')


In [380]:
null_cols = df.isnull().sum()
null_cols = null_cols[null_cols > 0]

print(null_cols)

Series([], dtype: int64)


In [383]:
print(df.shape)  # للتأكد من عدد الأعمدة بعد التنظيف
df.to_sql('silver_airbnb_master', con=engine, if_exists='replace', index=False)

(51707, 87)


11

## **EDA**

In [384]:
df


,realSum,room_type,room_shared,room_private,person_capacity,host_is_superhost,multi,biz,cleanliness_rating,guest_satisfaction_overall,bedrooms,dist,metro_dist,attr_index,attr_index_norm,rest_index,rest_index_norm,lng,lat,city,day_type,region,sub_region,Country,Population,Area in km2,attraction_count,nearest_attraction_km,nearest_attraction_name,restaurant_count,nearest_restaurant_km,nearest_restaurant_name,cafe_count,nearest_cafe_km,nearest_cafe_name,hotel_count,nearest_hotel_km,nearest_hotel_name,year,Approximate Annual Tourists,Currency,Majority Religion,Famous Foods,Language,Cost of Living,Safety,Cultural Significance,Description,city_avg_rating,city_avg_price,cost_score,price_vs_cost_of_living,park_count,nearest_park_km,nearest_park_name,hospital_count,nearest_hospital_km,nearest_hospital_name,airport_count,nearest_airport_km,nearest_airport_name,amenities_score,rating_score,location_score,city_popularity,Safety_Index,Crime_Index,Tourism_Season,Best_Months,property_type,accommodates,bathrooms,beds,host_total_listings_count,host_identity_verified,availability_365,number_of_reviews,reviews_per_month,review_scores_rating,review_scores_location,review_scores_cleanliness,amenities,amenities_count,wifi,parking,kitchen,air_conditioning
0,194.033698,Private room,False,True,2.0,False,1,0,10.0,93.0,1,5.022964,2.539380,78.690379,4.166708,98.253896,6.846473,4.90569,52.41772,amsterdam,weekdays,North Holland,Gemeente Landsmeer,Netherland,851573,219.32,0,5836.651235,Wilhelm Tell,55,0.058762,City Noord Eethuis,1,0.213570,Café de Bierhut,0,2.059126,DoubleTree by Hilton,2020,20 million,Euro,Christianity (Catholic & Protestant),"Stroopwafel, Herring, Bitterballen",Dutch,High,Moderately Safe,"Capital of the Netherlands, famous for canals,...",A vibrant European capital known for its histo...,94.514423,573.112795,4,48.508425,0,328.707590,Monks Farm Island,0,335.121223,St. George's Health & Wellbeing Hub,8,1.364320,Buiksloot Airfield,56,51.5,0.871521,56,70.2,29.8,Spring,Apr-Aug,Entire home,4.0,1.5,3.0,1.0,t,30.0,37.0,0.400000,4.650000,4.150000,4.920000,"[""Pocket wifi"", ""Private entrance"", ""Children\...",35,1,1,1,0
1,344.245776,Private room,False,True,4.0,False,0,0,8.0,85.0,1,0.488389,0.239404,631.176378,33.421209,837.280757,58.342928,4.90005,52.37432,amsterdam,weekdays,North Holland,Gemeente Amsterdam,Netherland,851573,219.32,0,5838.006367,Wilhelm Tell,1477,0.020374,Little Saigon,159,0.023696,Italiano,134,0.055198,Anco,2020,20 million,Euro,Christianity (Catholic & Protestant),"Stroopwafel, Herring, Bitterballen",Dutch,High,Moderately Safe,"Capital of the Netherlands, famous for canals,...",A vibrant European capital known for its histo...,94.514423,573.112795,4,86.061444,0,327.122457,Monks Farm Island,0,333.541628,St. George's Health & Wellbeing Hub,7,1.104106,Marine Kazerne Amsterdam Helipad,1636,46.5,1.953968,1770,70.2,29.8,Spring,Apr-Aug,Entire rental unit,6.0,1.5,6.0,11.0,t,216.0,65.0,2.690000,4.970000,4.940000,4.910000,"[""Radiant heating"", ""Private entrance"", ""Body ...",68,1,1,1,1
2,264.101422,Private room,False,True,2.0,False,0,1,9.0,87.0,1,5.748312,3.651621,75.275877,3.985908,95.386955,6.646700,4.97512,52.36103,amsterdam,weekdays,North Holland,Gemeente Diemen,Netherland,851573,219.32,0,5843.295571,Wilhelm Tell,87,0.022752,Vrijburcht,0,1.174662,Restaurant Zeeburg,1,0.940530,Amadi Panorama Hotel,2020,20 million,Euro,Christianity (Catholic & Protestant),"Stroopwafel, Herring, Bitterballen",Dutch,High,Moderately Safe,"Capital of the Netherlands, famous for canals,...",A vibrant European capital known for its histo...,94.514423,573.112795,4,66.025356,0,331.714118,Monks Farm Island,0,338.135932,St. George's Health & Wellbeing Hub,7,3.514870,Boels Verhuur Diemen Helipad,87,48.0,0.584654,88,70.2,29.8,Spring,Apr-Aug,Entire rental unit,4.0,1.0,2.0,1.0,t,75.0,54.0,0.480000,4.960000,4.690000,4.850000,"[""Bathtub"", ""Coffee maker"", ""Kitchen"", ""Shampo...",27,1,0,1,0
3,433.529398,Private room,False,True,4.0,False,0,1,9.0,90.0,2,0.384862,